# Special Topics in Software Engineering (TSIS) - Lab 1

Welcome to the first hands-on laboratory session for the **Topici Speciale în Inginerie Software** (_Special Topics in Software Engineering_ &mdash; TSIS) course! As mentioned in the first lecture, the objective of this course is to get you up to speed on **the latest advancements** in the field of [**large language models**](https://www.ibm.com/think/topics/large-language-models) (LLMs), focusing on the impact on developing [**agentic AI**](https://agentic.ai/what-is-agentic-ai) apps.

Unlike the first generation language models (like GPT-2, FLAN-T5, BERT etc.), newer models are more than simple chatbots; they are able to **call external tools** (access files, make API calls) and **structure their output** (to output Markdown, JSON, LaTeX etc.). Combining these powerful models with the appropriate framework, a **harness**, we can turn them into self-sufficient AI agents which are able to solve practical problems.

## Running the notebook

At the lab, we will work with the [Python](https://www.python.org/) language. Since most LLM providers expose a HTTP REST API, you could in theory query them using any programming language with a HTTP client. The agentic AI framework we'll use a bit later, [LangChain](https://www.langchain.com/), has official support for Python and JavaScript/TypeScript.

Therefore, please make sure you are comfortable wih using Python for working on the labs. For the final project, it's recommended to use Python or TypeScript for the "brain" of the AI agent (although nothing prevents you from calling into tools written in other languages).

## Gaining access to a hosted Large Language Model

Since we're interested in using Large Language Models and integrating them in our software products, we will not go into the nitty-gritty details of how to develop and train language models, or how to serve them efficiently in production (ask us if you want more resources on this). Instead, we will focus on how to use existing (frontier) large language models as end-users, through web APIs accessed by Python clients.

For the lab, we will work with [Mistral AI](https://mistral.ai/)'s free tier.

**Warning**: like most other LLM providers, Mistral's free-tier licensing mentions that they are allowed to use your prompts and data for training their own models. Therefore, do not include personally identifiable information or private data in your requests and be careful what you share with the provider!

## Installing dependencies

We will use [pip](https://pip.pypa.io/en/stable/installation/) to ensure the required packages are installed locally. If you're working locally, you might want to make sure you have configured a [virtual environment](https://docs.python.org/3/library/venv.html) beforehand.

In [ ]:
%pip install --upgrade mistralai pydantic

## Setting up the Mistral AI API client

We will use the official Mistral AI [SDK Client for Python](https://github.com/mistralai/client-python). The quickstart guide is available [here](https://docs.mistral.ai/resources/sdks?tab=python-sdk). Besides the examples included in the official client's docs, another helpful reference is the Mistral API documentation, available [here](https://docs.mistral.ai/api).

In [6]:
import os

api_key = os.environ.get("MISTRAL_API_KEY") or input("Enter your Mistral API key: ")
if not api_key:
    raise ValueError("MISTRAL_API_KEY is not set. Please provide a valid API key.")

ValueError: MISTRAL_API_KEY is not set. Please provide a valid API key.

In [ ]:
from mistralai import Mistral

client = Mistral(api_key=api_key)

In [ ]:
client.list_models()  # List available models

## Choosing the right model

https://docs.mistral.ai/inference/model-selection-guide

## Starting a conversation

In [ ]:
# Use ministral-8b as a baseline
MODEL = "ministral-8b"

response = client.chat.complete(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "Explain what the temperature parameter does for text generation with a Large Language Model.",
        },
    ],
    # Try playing with this!
    temperature=0.2,
)

print("Response:\n", response.choices[0].message.content)

print("\n--- Telemetry ---")
print(f"Prompt tokens: {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")

In [ ]:
class ConversationManager:
    def __init__(self, client: Mistral, system_prompt: str, max_history_turns: int = 3):
        self.client = client
        self.system_message = {"role": "system", "content": system_prompt}
        self.history = []  # Stores user/assistant pairs
        self.max_messages = max_history_turns * 2
        self.model = "mistral-medium-3-5"
        self.temperature = 0.2

    def chat(self, user_input: str) -> str:
        self.history.append({"role": "user", "content": user_input})

        # Trim oldest turns if exceeding window
        if len(self.history) > self.max_messages:
            self.history = self.history[-self.max_messages :]

        messages_payload = [self.system_message] + self.history

        res = self.client.chat.complete(
            model=self.model, messages=messages_payload, temperature=self.temperature
        )

        reply = res.choices[0].message.content
        self.history.append({"role": "assistant", "content": reply})
        print(
            f"[Debug] Active context messages: {len(messages_payload)} | Tokens: {res.usage.total_tokens}"
        )
        return reply

In [ ]:
# Let's test the sliding window
bot = ConversationManager(
    client, "You are a helpful SE tutor. Be brief.", max_history_turns=2
)
print(bot.chat("My project is written in Rust and uses Actix-web."))
print(bot.chat("What database library fits well with this stack?"))
print(bot.chat("How do I handle connection pooling there?"))
# By turn 4, turn 1 ("My project is written in Rust...") drops out of the window!
print(bot.chat("Remind me, what web framework did I say I was using?"))

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class BugTriageReport(BaseModel):
    summary: str = Field(description="Concise 1-sentence technical summary of the issue")
    severity: Literal["LOW", "MEDIUM", "HIGH", "CRITICAL"]
    affected_component: Literal["auth", "database", "frontend", "payment", "unknown"]
    reproduction_steps: list[str] = Field(description="Ordered list of steps to reproduce")
    requires_immediate_rollback: bool

raw_ticket = """
Hey team, ever since the v2.4 deploy 10 minutes ago, users in EU cannot complete checkout!
When they click 'Pay Now' with Stripe, the spinner hangs forever and the console logs a 502 Bad Gateway
on /api/v2/charge. Tested on Chrome and Firefox. We are losing orders right now!
"""

# client.chat.parse automatically converts the Pydantic model to JSON Schema
# and validates the response back into a Python object.
parsed_response = client.chat.parse(
    model="mistral-small-latest",
    messages=[
        {"role": "system", "content": "Extract structured bug triage data from raw incident reports."},
        {"role": "user", "content": raw_ticket}
    ],
    response_format=BugTriageReport,
    temperature=0.0  # Always 0.0 for data extraction
)

triage_obj: BugTriageReport = parsed_response.choices[0].message.parsed
print(f"Type: {type(triage_obj)}")
print(f"Severity: {triage_obj.severity} | Rollback? {triage_obj.requires_immediate_rollback}")
print(f"Component: {triage_obj.affected_component}")
print(f"Steps: {triage_obj.reproduction_steps}")

## The Tool Age

In [ ]:
import json

# 1. Define the actual local Python functions
def get_service_health(service_name: str) -> str:
    mock_db = {
        "auth-service": {"status": "healthy", "latency_ms": 42, "error_rate": 0.01},
        "payment-gateway": {"status": "degraded", "latency_ms": 1850, "error_rate": 14.5},
    }
    return json.dumps(mock_db.get(service_name, {"status": "not_found"}))

# 2. Define the JSON Schema contract for the LLM
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_service_health",
            "description": "Fetch live telemetry and health status for a production microservice.",
            "parameters": {
                "type": "object",
                "properties": {
                    "service_name": {
                        "type": "string",
                        "enum": ["auth-service", "payment-gateway"],
                        "description": "The exact identifier of the microservice."
                    }
                },
                "required": ["service_name"]
            }
        }
    }
]

messages = [
    {"role": "system", "content": "You are an SRE assistant. Check live telemetry before answering."},
    {"role": "user", "content": "Why is checkout failing right now? Check the payment gateway."}
]

# Step A: First call — LLM decides to call the tool
response = client.chat.complete(
    model="mistral-small-latest",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

assistant_msg = response.choices[0].message
messages.append(assistant_msg)

# Step B: Check if the LLM requested a tool execution
if assistant_msg.tool_calls:
    for tool_call in assistant_msg.tool_calls:
        fn_name = tool_call.function.name
        fn_args = json.loads(tool_call.function.arguments)
        print(f"[Runtime] LLM requested tool '{fn_name}' with args: {fn_args}")

        # Execute local Python code
        if fn_name == "get_service_health":
            tool_output = get_service_health(**fn_args)

        # Append the observation back into the message history
        messages.append({
            "role": "tool",
            "name": fn_name,
            "content": tool_output,
            "tool_call_id": tool_call.id
        })

    # Step C: Second call — LLM synthesizes the final response using the tool output
    final_response = client.chat.complete(
        model="mistral-small-latest",
        messages=messages
    )
    print("\nFinal SRE Report:\n", final_response.choices[0].message.content)

## 6. Tool calling: the application remains in control

Tool calling is not the model executing Python. The model chooses a function and supplies arguments; **our application validates and executes the function**, then sends the result back as a tool message. The model can then produce a final answer using that observation.

In [ ]:
import json


def get_service_health(service_name: str) -> str:
    """Return simulated telemetry for an allow-listed service."""
    telemetry = {
        "auth-service": {"status": "healthy", "latency_ms": 42, "error_rate_percent": 0.01},
        "payment-gateway": {"status": "degraded", "latency_ms": 1850, "error_rate_percent": 14.5},
    }
    if service_name not in telemetry:
        raise ValueError(f"Unknown service: {service_name}")
    return json.dumps(telemetry[service_name])


tools = [
    {
        "type": "function",
        "function": {
            "name": "get_service_health",
            "description": "Read current health telemetry for an allow-listed production service.",
            "parameters": {
                "type": "object",
                "properties": {
                    "service_name": {
                        "type": "string",
                        "enum": ["auth-service", "payment-gateway"],
                    }
                },
                "required": ["service_name"],
            },
        },
    }
]

messages = [
    {"role": "system", "content": "You are an SRE assistant. Inspect telemetry before explaining incidents."},
    {"role": "user", "content": "Why is checkout failing? Check the payment gateway and summarize the evidence."},
]

# Call 1: the model may request a tool, but it cannot execute it itself.
first = client.chat.complete(model=MODEL, messages=messages, tools=tools, tool_choice="auto")
assistant_message = first.choices[0].message
messages.append(assistant_message)

if not assistant_message.tool_calls:
    print(assistant_message.content)
else:
    for tool_call in assistant_message.tool_calls:
        if tool_call.function.name != "get_service_health":
            raise ValueError(f"Refusing unknown tool: {tool_call.function.name}")
        arguments = json.loads(tool_call.function.arguments)
        print(f"[application] calling {tool_call.function.name}({arguments})")
        observation = get_service_health(**arguments)
        messages.append({
            "role": "tool",
            "name": tool_call.function.name,
            "content": observation,
            "tool_call_id": tool_call.id,
        })

    # Call 2: the model receives the observation and writes the user-facing answer.
    final = client.chat.complete(model=MODEL, messages=messages, temperature=0.2, max_tokens=180)
    print("\nFinal report:\n" + final.choices[0].message.content)

### Exercise 2 — extend the tool safely

Add a second read-only tool, such as `get_recent_deploys()`, and expose it in the schema. Keep the implementation deterministic and allow-listed. Then ask a question that requires both tools. What happens if the model requests a tool or argument that your application does not recognize?

### Exercise 3 — mini design challenge

Design a small agent for a software engineering task (for example, a test-failure triager). Specify:

1. the system instruction and user input;
2. the structured result you would validate;
3. one or two tools, including their safety checks; and
4. one failure mode and how a human or ordinary code would detect it.

Do not implement a tool that deletes data, changes production, or sends messages.

## Takeaways

- An LLM API is a probabilistic component, not a database or an authority.
- Context is explicit: the application decides what history to send.
- Structured output gives us a machine-checkable interface, but not guaranteed truth.
- Tool calling is an application-controlled loop: request → validate → execute → observe → respond.
- Least privilege, allow-lists, timeouts, logging, and human approval matter more as tools become capable.

Before leaving, save your answers to Exercises 1–3 and delete any local copies of API keys. In the next laboratory we will build on this loop to make an agent more reliable and observable.